In [44]:
import torch.nn.functional as F

def up(inp,target):
    return F.interpolate(inp,size=target.shape[2:],mode="bilinear",align_corners=True)
#this i have odne to match shape as x11 and x02 or others when upgrade have different image size 
#hence causes issue

In [45]:
import torch
class Encoder(torch.nn.Module):
    def __init__(self,i_channel,o_channel,kernel=3):
        super().__init__()
        self.Seq=torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=i_channel,
                            kernel_size=kernel,
                            out_channels=o_channel,
                            padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=o_channel,
                            kernel_size=kernel,
                            out_channels=o_channel,
                            padding=1),
            torch.nn.ReLU()
        )
        self.Pool=torch.nn.MaxPool2d(kernel_size=2,
                                     stride=2)
        #essa esliya liya kyunki Seq ke output ko middle layers lo bhi pass
        #karna hai aur lower level ko bhi aur lower level me bass pool chahiye side me nahi
    def forward(self,inp):
        skip=self.Seq(inp)
        out=self.Pool(skip)
        return out,skip



In [ ]:
class Decoder(torch.nn.Module):
    def __init__(self,i_channel,skip_channel,o_channel,kernel=3):
        super().__init__()
        self.Seq1=torch.nn.Sequential(
               torch.nn.Upsample(scale_factor=2,
                                 mode='bilinear',
                                 align_corners=True),
               torch.nn.Conv2d(in_channels=i_channel,
                               out_channels=o_channel,
                               kernel_size=kernel,
                               padding=1),
               torch.nn.ReLU()
        )
        self.Seq2=torch.nn.Sequential(
               torch.nn.Conv2d(in_channels=o_channel+skip_channel,
                               out_channels=o_channel,
                               kernel_size=kernel,
                               padding=1),
               torch.nn.ReLU(),
               torch.nn.Conv2d(in_channels=o_channel,
                               out_channels=o_channel,
                               kernel_size=kernel,
                               padding=1),
               torch.nn.ReLU()
        )
    def forward(self,inp,skips):
        upgrade=self.Seq1(inp)
        con = torch.cat(list(upgrade).extend(skips), dim=1)
        out = self.Seq2(con)
        return out

In [47]:
class Midlayer(torch.nn.Module):
    def __init__(self, i_channel, o_channel,kernel=3):
        super().__init__()
        self.Seq=torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=i_channel,out_channels=o_channel,kernel_size=kernel,padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=o_channel,out_channels=o_channel,kernel_size=kernel,padding=1),
            torch.nn.ReLU()
        )
    def forward(self,inputs):
        con_cat=torch.cat(inputs,dim=1)
        return self.Seq(con_cat)

In [48]:
'''
layer1_out_channel=64,
layer3_out_channel=128,
layer3_out_channel=256,
layer4_out_channel=512,
final_layer_out_channel=1024,
'''
class UnetPP(torch.nn.Module):
    def __init__(self,):
        super().__init__()
        self.X00=Encoder(3,64)
        self.X10=Encoder(64,128)
        self.X01=Midlayer(64+128,64)      #i_channel=X00+X10 o_channel=layer_out_channels)
        self.X20=Encoder(128,256)
        self.X11=Midlayer(128+256,128)
        self.X02=Midlayer(64+64+128,64)    #i_channel=X00+X11,X01 o_channel=layer2_out_channels)
        self.X30=Encoder(256,512)
        self.X21=Midlayer(256+512,256)
        self.X12=Midlayer(128+256+128,128)
        self.X03=Midlayer(64+64+64+128,64)
        self.X40=torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=512,
                            kernel_size=3,
                            out_channels=1024,
                            padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=1024,
                            kernel_size=3,
                            out_channels=1024,
                            padding=1),
            torch.nn.ReLU()
        )
        self.X31=Decoder(1024,512,512)
        self.X22=Decoder(512,256*2,256)
        self.X13=Decoder(256,128*3,128)
        self.X04=Decoder(128,64*4,64)
        self.final=torch.nn.Conv2d(in_channels=64,
                                    out_channels=1,
                                    kernel_size=1,
                                    padding=0)
    def forward(self,inp):
        out,skip0_0=self.X00(inp)
        out,skip1_0=self.X10(out)
        skip0_1=self.X01([skip0_0,up(skip1_0,skip0_0)])
        out,skip2_0=self.X20(out)
        skip1_1=self.X11([skip1_0,up(skip2_0,skip1_0)])
        skip0_2=self.X02([skip0_0,skip0_1,up(skip1_1,skip0_1)])
        out,skip3_0=self.X30(out)
        skip2_1=self.X21([skip2_0,up(skip3_0,skip2_0)])
        skip1_2=self.X12([skip1_0,skip1_1,up(skip2_1,skip1_1)])
        skip0_3=self.X03([skip0_0,skip0_2,skip0_1,up(skip1_2,skip0_2)])
        out=self.X40(out)
        out=self.X31(out,[skip3_0])
        out=self.X22(out,[skip2_1,skip2_0])
        out=self.X13(out,[skip1_2,skip1_1,skip1_0])
        out=self.X04(out,[skip0_3,skip0_2,skip0_1,skip0_0])
        out=self.final(out)
        return out

In [49]:
# x = torch.randn(1, 3, 256, 256)
# model = UnetPP()                    -->>>> to check shape is correct or not
# y = model(x)
# print(y.shape)

In [50]:
class Dice(torch.nn.Module):
    def __init__(self,):
        super().__init__
        self.smooth=1e-6
    def forward(self,pred,target):
        pred=torch.sigmoid(pred)
        pred=pred.view(-1)
        target=target.view(-1)
        intersect=(target*pred).sum()
        return (2*intersect)+self.smooth/pred.sum()+target.sum()+self.smooth
    
BCE=torch.nn.BCEWithLogitsLoss()
dice=Dice()

def Loss(pred,target):
    return 0.3*BCE(pred,target)+0.7*(pred,target)

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(".."))
loader=DataLoader(dataset,batch_size=16,shuffle=True)


In [68]:
import tqdm
device="cuda" if torch.cuda.is_available() else "cpu"
model=UnetPP().to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)
Train=True
if Train:
    total_epochs=10
    total_batches=total_epochs*len(loader)
    with tqdm.tqdm(total=total_batches) as b:
        for epoch in range(total_epochs):
            model.train()
            for img,mask in loader:
                img=img.to(device)
                mask=mask.to(device)
                pred=model(img)
                loss=Loss(pred,mask)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_loss+=loss
                b.update(1)
            print(f"Total epochs :{epchos+1} Loss :{total_loss/len(loader):.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 10.76 GiB is allocated by PyTorch, and 32.24 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)